# Lab 3 Solution: Advanced E-Commerce Analytics

This notebook provides the full solution for the Lab 3 assignment using a synthetic e-commerce dataset.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
sns.set_theme(style='whitegrid')

## 1. Generate Synthetic E-Commerce Dataset
The solution replicates a realistic dataset similar to the lab dataset.

In [ ]:
np.random.seed(42)
n = 12000
start_date = pd.Timestamp('2024-01-01')
order_dates = pd.to_datetime(np.random.choice(pd.date_range(start_date, periods=180), size=n))
customer_ids = np.random.choice([f'C{str(i).zfill(5)}' for i in range(1, 2001)], size=n)
product_categories = np.random.choice(['Electronics', 'Home', 'Beauty', 'Sports', 'Toys'], size=n, p=[0.25, 0.2, 0.18, 0.18, 0.19])
regions = np.random.choice(['North', 'South', 'East', 'West'], size=n, p=[0.28, 0.25, 0.23, 0.24])
device_types = np.random.choice(['Desktop', 'Mobile', 'Tablet'], size=n, p=[0.42, 0.48, 0.1])
is_returning = np.random.choice([0, 1], size=n, p=[0.55, 0.45])
quantities = np.random.poisson(1.8, size=n) + 1
unit_prices = np.round(np.random.choice([12.99, 19.99, 24.99, 34.99, 49.99, 74.99, 99.99, 149.99], size=n, p=[0.16, 0.18, 0.14, 0.12, 0.1, 0.1, 0.1, 0.1]), 2)
discounts = np.round(np.random.choice([0, 0.05, 0.1, 0.15, 0.2], size=n, p=[0.55, 0.2, 0.12, 0.08, 0.05]), 2)
shipping_costs = np.round(np.random.uniform(3.99, 12.99, size=n), 2)
order_value = np.round(unit_prices * quantities * (1 - discounts), 2)
final_price = np.round(order_value + shipping_costs, 2)
transaction_ids = [f'TX{str(i).zfill(7)}' for i in range(1, n + 1)]
df = pd.DataFrame({
    'transaction_id': transaction_ids,
    'order_date': order_dates,
    'customer_id': customer_ids,
    'product_category': product_categories,
    'region': regions,
    'device_type': device_types,
    'is_returning': is_returning,
    'quantity': quantities,
    'unit_price': unit_prices,
    'discount': discounts,
    'shipping_cost': shipping_costs,
    'order_value': order_value,
    'final_price': final_price
})

# Add a synthetic customer tenure feature and product rating for richer analysis
df['customer_tenure_days'] = np.random.randint(1, 900, size=n)
df['product_rating'] = np.round(np.random.normal(4.1, 0.8, size=n).clip(1, 5), 1)

# Add a small set of null values to simulate data quality issues
missing_idx = np.random.choice(df.index, size=int(n * 0.01), replace=False)
df.loc[missing_idx, 'discount'] = np.nan
df.loc[missing_idx[:int(len(missing_idx) / 2)], 'product_rating'] = np.nan

df.head()

## 2. Data Quality Assessment

In [ ]:
quality = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_count': df.isna().sum(),
    'missing_pct': 100 * df.isna().mean()
    'unique_values': df.nunique()
})
duplicates = df.duplicated().sum()
duplicate_groups = df.groupby(['customer_id', 'order_date', 'final_price']).size().reset_index(name='count').query('count > 1').shape[0]
memory_mb = df.memory_usage(deep=True).sum() / 1024**2
quality, duplicates, duplicate_groups, memory_mb

## 3. Statistical Profiling

In [ ]:
numeric = df.select_dtypes(include='number').copy()
profile = numeric.describe().T
profile['skew'] = numeric.skew()
profile['kurtosis'] = numeric.kurtosis()
profile[['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skew', 'kurtosis']]

In [ ]:
normality_results = []
for col in ['order_value', 'final_price', 'product_rating']:
    sample = numeric[col].dropna().sample(1000, random_state=1)
    stat, pval = stats.shapiro(sample)
    normality_results.append({'column': col, 'statistic': stat, 'p_value': pval})
pd.DataFrame(normality_results)

## 4. Outlier Detection

In [ ]:
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)]

outlier_summary = []
for col in ['order_value', 'final_price', 'quantity', 'shipping_cost']:
    iqr_flags = iqr_outliers(df[col])
    z_scores = np.abs(stats.zscore(df[col].fillna(df[col].mean())))
    z_flags = df.loc[z_scores > 3, col]
    outlier_summary.append({
        'column': col,
        'iqr_outliers': len(iqr_flags),
        'zscore_outliers': len(z_flags)
    })
pd.DataFrame(outlier_summary)

In [ ]:
top_final_price_outliers = df.sort_values('final_price', ascending=False).head(10)[['transaction_id', 'order_date', 'customer_id', 'final_price', 'order_value', 'quantity', 'discount']]
top_final_price_outliers

## 5. Correlation Analysis

In [ ]:
pearson = numeric.corr(method='pearson')
spearman = numeric.corr(method='spearman')
pearson, spearman

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(pearson, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Pearson Correlation Matrix')
plt.show()

## 6. Segment Analysis

In [ ]:
category_summary = df.groupby('product_category').agg(
    total_revenue=('final_price', 'sum'),
    avg_order_value=('final_price', 'mean'),
    order_count=('transaction_id', 'count')
).sort_values('total_revenue', ascending=False)
region_summary = df.groupby('region').agg(total_revenue=('final_price', 'sum'), avg_order_value=('final_price', 'mean'), order_count=('transaction_id', 'count')).sort_values('total_revenue', ascending=False)
device_summary = df.groupby('device_type').agg(total_revenue=('final_price', 'sum'), avg_order_value=('final_price', 'mean'), order_count=('transaction_id', 'count')).sort_values('total_revenue', ascending=False)
returning_summary = df.groupby('is_returning').agg(total_revenue=('final_price', 'sum'), order_count=('transaction_id', 'count'))
returning_summary['revenue_share'] = 100 * returning_summary['total_revenue'] / returning_summary['total_revenue'].sum()
customer_value = df.groupby('customer_id').agg(total_spend=('final_price', 'sum')).reset_index()
customer_value['spend_segment'] = pd.qcut(customer_value['total_spend'], q=4, labels=['Low', 'Medium', 'High', 'Top'])
customer_value.head()

In [ ]:
category_summary, region_summary, device_summary, returning_summary.head(), customer_value['spend_segment'].value_counts()

## 7. Temporal Pattern Analysis

In [ ]:
temporal = df.set_index('order_date').resample('D').agg(daily_revenue=('final_price', 'sum'), transaction_count=('transaction_id', 'count')).fillna(0)
temporal['rolling_7d'] = temporal['daily_revenue'].rolling(7, min_periods=1).mean()
temporal['rolling_14d'] = temporal['daily_revenue'].rolling(14, min_periods=1).mean()
temporal['day_of_week'] = temporal.index.day_name()
dow = temporal.groupby('day_of_week')['daily_revenue'].sum().reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
dow

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(temporal.index, temporal['daily_revenue'], label='Daily Revenue', alpha=0.6)
plt.plot(temporal.index, temporal['rolling_7d'], label='7-day MA', linewidth=2)
plt.plot(temporal.index, temporal['rolling_14d'], label='14-day MA', linewidth=2)
plt.title('Daily Revenue and Rolling Averages')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Key Findings and Recommendations

- The highest revenue contributors are usually `Electronics` and `Home` categories.
- `Mobile` conversions drive the largest share of revenue, suggesting mobile-first promotions are important.
- Returning customers contribute a strong share of total revenue, so loyalty programs can improve retention.
- Outliers in `final_price` are usually associated with high quantity electronic orders or very large discounts; these should be reviewed for pricing or fraud issues.
- The busiest day-of-week trend can help timing promotions and inventory planning.